# LeetCode 915 — Partition Array into Disjoint Intervals bằng Python

Link đề: https://leetcode.com/problems/partition-array-into-disjoint-intervals/ · Độ khó: **Medium** · Chủ đề: Array (mảng)

Skeleton LeetCode:

```python
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        ...
```

Ghi chú: ta cắt `nums` thành hai đoạn liên tiếp `left` và `right` (cả hai khác rỗng) sao cho **mọi phần tử của `left` nhỏ hơn hoặc bằng mọi phần tử của `right`**, và `left` ngắn nhất có thể. Điều kiện này tương đương `max(left) <= min(right)`. Dùng **max tiền tố** và **min hậu tố** ta kiểm tra mỗi điểm cắt trong `O(1)`; hoặc quét **một lượt** với hai biến: max của `left` hiện tại và max toàn cục đã gặp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm độ dài nhỏ nhất của `left` sao cho `max(left) <= min(right)` (đề **đảm bảo** có đáp án hợp lệ).
2. Nắm các công cụ Python cần thiết: mảng max tiền tố và min hậu tố, vòng lặp ngược `range(n - 2, -1, -1)`, `max`/`min`, và `collections.Counter` / `defaultdict` để so sánh đa tập.
3. Hiểu nguyên lý: một điểm cắt hợp lệ khi và chỉ khi **max của tiền tố** `<=` **min của hậu tố**; điểm cắt hợp lệ **nhỏ nhất** cho `left` ngắn nhất; có thể xác định bằng quét một lượt.
4. Đi qua 4 cách: thử từng điểm cắt với quét (brute), max tiền tố và min hậu tố, quét một lượt hai biến (cách nộp chính), và so sánh tiền tố của `nums` với tiền tố của bản đã sắp xếp bằng bộ đếm; đối chiếu bằng oracle.
5. Tránh các lỗi kinh điển: dùng `<` thay `<=`, quên cập nhật max của `left` sau khi mở rộng, so sánh với mảng đã sắp xếp chỉ qua `max`, và dùng thuật toán `O(n^2)` với `n = 10^5`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, hãy chia nó thành hai mảng con **liên tiếp** `left` và `right` sao cho:

- mọi phần tử của `left` **nhỏ hơn hoặc bằng** mọi phần tử của `right`;
- `left` và `right` đều **khác rỗng**;
- `left` có kích thước **nhỏ nhất** có thể.

Trả về **độ dài của `left`** sau khi chia. Các test được tạo sao cho luôn tồn tại cách chia.

**Ví dụ 1**

- Input: `nums = [5,0,3,8,6]`
- Output: `3`
- Giải thích: `left = [5,0,3]`, `right = [8,6]`.

**Ví dụ 2**

- Input: `nums = [1,1,1,0,6,12]`
- Output: `4`
- Giải thích: `left = [1,1,1,0]`, `right = [6,12]`.

**Ràng buộc:**

- `2 <= nums.length <= 10^5`
- `0 <= nums[i] <= 10^6`
- Có ít nhất một đáp án hợp lệ cho đầu vào.

In [1]:
from typing import List
from collections import Counter, defaultdict
import random, time

examples = [
    ([5, 0, 3, 8, 6], 3),
    ([1, 1, 1, 0, 6, 12], 4),
]
for nums, expected in examples:
    print(f"nums={nums} kỳ vọng={expected}")

nums=[5, 0, 3, 8, 6] kỳ vọng=3
nums=[1, 1, 1, 0, 6, 12] kỳ vọng=4


## 3. Kiến thức nền

### 3.1. Điều kiện "mọi phần tử trái `<=` mọi phần tử phải"

"Mọi `x` trong `left` và mọi `y` trong `right` đều thỏa `x <= y`" tương đương `max(left) <= min(right)`: phần tử lớn nhất bên trái không vượt phần tử nhỏ nhất bên phải. Nên chỉ cần biết `max` của tiền tố và `min` của hậu tố tại điểm cắt.

In [2]:
def valid_split(nums: List[int], k: int) -> bool:
    """left = nums[:k], right = nums[k:] (1 <= k <= n - 1)."""
    return max(nums[:k]) <= min(nums[k:])

nums = [5, 0, 3, 8, 6]
for k in range(1, len(nums)):
    print(f"k={k}: left={nums[:k]}, right={nums[k:]}, max(left)={max(nums[:k])}, min(right)={min(nums[k:])}, hợp lệ? {valid_split(nums, k)}")

k=1: left=[5], right=[0, 3, 8, 6], max(left)=5, min(right)=0, hợp lệ? False
k=2: left=[5, 0], right=[3, 8, 6], max(left)=5, min(right)=3, hợp lệ? False
k=3: left=[5, 0, 3], right=[8, 6], max(left)=5, min(right)=6, hợp lệ? True
k=4: left=[5, 0, 3, 8], right=[6], max(left)=8, min(right)=6, hợp lệ? False


### 3.2. Max tiền tố và min hậu tố

- `pm[i] = max(nums[0..i])` (max tiền tố), tính tiến: `pm[i] = max(pm[i - 1], nums[i])`.
- `sm[i] = min(nums[i..n-1])` (min hậu tố), tính lùi: `sm[i] = min(sm[i + 1], nums[i])`.

Điểm cắt với `left = nums[:k]` hợp lệ khi `pm[k - 1] <= sm[k]`.

In [3]:
def prefix_max(nums):
    pm = []
    cur = None
    for x in nums:
        cur = x if cur is None else max(cur, x)
        pm.append(cur)
    return pm

def suffix_min(nums):
    n = len(nums)
    sm = [0] * n
    sm[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        sm[i] = min(sm[i + 1], nums[i])
    return sm

nums = [5, 0, 3, 8, 6]
print("max tiền tố:", prefix_max(nums))
print("min hậu tố :", suffix_min(nums))

max tiền tố: [5, 5, 5, 8, 8]
min hậu tố : [0, 0, 3, 6, 6]


### 3.3. Điểm cắt hợp lệ nhỏ nhất

Ta cần `k` nhỏ nhất thỏa điều kiện, nên **quét `k` từ `1` tăng dần** và trả về điểm cắt hợp lệ đầu tiên. Đề đảm bảo tồn tại (ít nhất `k = n - 1`... không nhất thiết, nhưng có một điểm hợp lệ nào đó).

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `nums = [5, 0, 3, 8, 6]`.

| `k` | `left` | `right` | `max(left)` | `min(right)` | hợp lệ? |
|---|---|---|---|---|---|
| 1 | `[5]` | `[0,3,8,6]` | 5 | 0 | không |
| 2 | `[5,0]` | `[3,8,6]` | 5 | 3 | không |
| 3 | `[5,0,3]` | `[8,6]` | 5 | 6 | **có** |
| 4 | `[5,0,3,8]` | `[6]` | 8 | 6 | không |

`k = 3` là điểm hợp lệ nhỏ nhất. Ví dụ 2: `[1,1,1,0,6,12]`: `k = 1,2,3` có `right` chứa `0` nhỏ hơn `max(left) = 1`; `k = 4`: `left = [1,1,1,0]`, `max = 1 <= min(right) = 6`: hợp lệ.

## 5. Cách 1 — Brute Force: thử từng điểm cắt bằng quét

**Ý tưởng.** Với mỗi `k = 1..n-1`: kiểm tra từng cặp hoặc tính `max(nums[:k])` và `min(nums[k:])` bằng quét `O(n)`; trả `k` đầu tiên thỏa điều kiện. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [4]:
def partition_brute(nums: List[int]) -> int:
    n = len(nums)
    for k in range(1, n):
        if all(x <= y for x in nums[:k] for y in nums[k:]):       # kiểm tra mọi cặp (định nghĩa gốc)
            return k
    return -1

for nums, expected in examples:
    got = partition_brute(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 0, 3, 8, 6] -> 3 OK
[1, 1, 1, 0, 6, 12] -> 4 OK


**Độ phức tạp và nút thắt.** Kiểm tra mọi cặp có thể tốn `O(n^2)` mỗi điểm cắt (tổng `O(n^3)`) khi cặp vi phạm nằm cuối; ngay cả bản tính `max(nums[:k])` và `min(nums[k:])` bằng quét cũng cần `O(n)` mỗi điểm cắt, tổng `O(n^2)`. Với `n = 10^5` là `~ 10^10` thao tác: không thể. Hãy xem thời gian của bản quét `max`/`min` khi điểm cắt hợp lệ nằm xa (phải thử gần hết các `k`):

In [5]:
def partition_scan_minmax(nums: List[int]) -> int:
    for k in range(1, len(nums)):
        if max(nums[:k]) <= min(nums[k:]):
            return k
    return -1

for n in (500, 1000, 2000, 4000):
    nums = [n] + list(range(n - 1)) + [n + 1]            # điểm cắt hợp lệ đầu tiên là k = n
    t0 = time.perf_counter()
    r = partition_scan_minmax(nums)
    print(f"n={len(nums):5}: kết quả {r}, {1000 * (time.perf_counter() - t0):9.1f} ms")

n=  501: kết quả 500,       3.7 ms
n= 1001: kết quả 1000,      10.7 ms
n= 2001: kết quả 2000,      45.6 ms


n= 4001: kết quả 4000,     176.1 ms


Thời gian tăng gấp bốn lần khi `n` tăng gấp đôi (`O(n^2)`). Nút thắt: ta tính lại `max` của tiền tố và `min` của hậu tố **từ đầu** ở mỗi điểm cắt, trong khi chúng thay đổi **từng bước một**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tính trước max tiền tố và min hậu tố** để mỗi điểm cắt kiểm tra trong `O(1)`; hoặc tiến xa hơn: **quét một lượt** và duy trì điểm cắt hợp lệ nhỏ nhất hiện tại.

**Quét một lượt.** Giữ `left_max` (max của `left` hiện tại), `global_max` (max của mọi phần tử đã duyệt) và `pos` (độ dài `left` hiện tại). Với phần tử mới `x = nums[i]`:

- cập nhật `global_max = max(global_max, x)`;
- nếu `x < left_max`: `x` **phải** thuộc `left` (vì `right` không được chứa phần tử nhỏ hơn `max(left)`), nên mở rộng `left` tới hết `i`: `pos = i + 1`, `left_max = global_max` (vì `left` nay chứa mọi phần tử đã duyệt);
- ngược lại để `x` ở `right` (tạm thời).

| Cách | Ý tưởng |
|---|---|
| 2 | Max tiền tố + min hậu tố: tìm `k` nhỏ nhất với `pm[k-1] <= sm[k]` |
| 3 | Quét một lượt hai biến (nộp) |
| 4 | So tiền tố của `nums` với tiền tố của bản **đã sắp xếp** bằng bộ đếm |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Max tiền tố và min hậu tố

**Giải thích từng đoạn.**

- `pm[i]` là max của `nums[0..i]`, tính một lượt tiến.
- `sm[i]` là min của `nums[i..n-1]`, tính một lượt lùi.
- Duyệt `k` từ `1` đến `n - 1`: nếu `pm[k - 1] <= sm[k]` thì trả `k`.

In [6]:
def partition_prefix_suffix(nums: List[int]) -> int:
    n = len(nums)
    pm = [0] * n
    pm[0] = nums[0]
    for i in range(1, n):
        pm[i] = max(pm[i - 1], nums[i])
    sm = [0] * n
    sm[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        sm[i] = min(sm[i + 1], nums[i])
    for k in range(1, n):
        if pm[k - 1] <= sm[k]:
            return k
    return -1

for nums, expected in examples:
    got = partition_prefix_suffix(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 0, 3, 8, 6] -> 3 OK
[1, 1, 1, 0, 6, 12] -> 4 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        n = len(nums)
        pm = [0] * n
        pm[0] = nums[0]
        for i in range(1, n):
            pm[i] = max(pm[i - 1], nums[i])
        sm = [0] * n
        sm[-1] = nums[-1]
        for i in range(n - 2, -1, -1):
            sm[i] = min(sm[i + 1], nums[i])
        for k in range(1, n):
            if pm[k - 1] <= sm[k]:
                return k
        return -1

sol_prefix = Solution().partitionDisjoint
assert [sol_prefix(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n)` thời gian, `O(n)` bộ nhớ (hai mảng phụ).

---

### 7.2. Cách 3 — Quét một lượt với hai biến (cách nộp chính)

**Giải thích từng đoạn.**

- Khởi tạo: `left_max = global_max = nums[0]`, `pos = 1` (`left` ban đầu chỉ chứa `nums[0]`).
- Với `i` từ `1` đến `n - 1`: `global_max = max(global_max, nums[i])`; nếu `nums[i] < left_max` thì `left_max = global_max` và `pos = i + 1`.
- Trả `pos`.

Vì sao `left_max = global_max` khi mở rộng: `left` giờ gồm `nums[0..i]`, max của nó chính là max toàn cục đã duyệt.

In [8]:
def partition_one_pass(nums: List[int]) -> int:
    left_max = global_max = nums[0]
    pos = 1
    for i in range(1, len(nums)):
        global_max = max(global_max, nums[i])
        if nums[i] < left_max:
            left_max = global_max
            pos = i + 1
    return pos

for nums, expected in examples:
    got = partition_one_pass(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 0, 3, 8, 6] -> 3 OK
[1, 1, 1, 0, 6, 12] -> 4 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        left_max = global_max = nums[0]
        pos = 1
        for i in range(1, len(nums)):
            global_max = max(global_max, nums[i])
            if nums[i] < left_max:
                left_max = global_max
                pos = i + 1
        return pos

sol_one = Solution().partitionDisjoint
assert [sol_one(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

---

### 7.3. Cách 4 — So tiền tố của `nums` với tiền tố của bản đã sắp xếp

**Ý tưởng khác.** Nếu `left = nums[:k]` thỏa mọi phần tử `<=` mọi phần tử của `right`, thì `left` chính là **`k` phần tử nhỏ nhất** (đa tập) của `nums`, tức đa tập của `nums[:k]` **bằng** đa tập của `sorted(nums)[:k]`. Ngược lại, nếu hai đa tập này bằng nhau thì `left` gồm `k` phần tử nhỏ nhất nên mọi phần tử của `left` `<=` mọi phần tử của `right`. Ta duy trì **hiệu số đếm** `diff[v] = (số lần v trong nums[:k]) - (số lần v trong sorted[:k])` và số khóa khác `0`; điểm cắt hợp lệ khi số khóa khác `0` bằng `0`.

**Giải thích từng đoạn.** Mỗi bước `k`: cộng `1` cho `nums[k - 1]` và trừ `1` cho `s[k - 1]` (với `s` là bản sắp xếp), cập nhật `nonzero` khi một khóa chuyển giữa `0` và khác `0`.

In [10]:
def partition_sorted_prefix(nums: List[int]) -> int:
    s = sorted(nums)                                      # cách này (và chỉ cách này) dùng sắp xếp
    diff = defaultdict(int)
    nonzero = 0

    def bump(v, d):
        nonlocal nonzero
        before = diff[v] != 0
        diff[v] += d
        after = diff[v] != 0
        nonzero += after - before

    for k in range(1, len(nums)):
        bump(nums[k - 1], +1)
        bump(s[k - 1], -1)
        if nonzero == 0:
            return k
    return -1

for nums, expected in examples:
    got = partition_sorted_prefix(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 0, 3, 8, 6] -> 3 OK
[1, 1, 1, 0, 6, 12] -> 4 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        s = sorted(nums)
        diff = {}
        nonzero = 0
        for k in range(1, len(nums)):
            for v, d in ((nums[k - 1], 1), (s[k - 1], -1)):
                before = diff.get(v, 0)
                diff[v] = before + d
                nonzero += (diff[v] != 0) - (before != 0)
            if nonzero == 0:
                return k
        return -1

sol_sorted = Solution().partitionDisjoint
assert [sol_sorted(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n log n)` thời gian (sắp xếp), `O(n)` bộ nhớ. Chậm hơn hai cách trước nhưng độc lập với việc dùng `max`/`min`, nên là một kiểm chứng tốt.

## 8. TRACE từng bước

### 8.1. Cách 3 (quét một lượt) cho ví dụ 1: `[5, 0, 3, 8, 6]`

In [12]:
def trace_one_pass(nums):
    left_max = global_max = nums[0]
    pos = 1
    print(f"khởi tạo: left_max={left_max}, global_max={global_max}, pos={pos}")
    for i in range(1, len(nums)):
        global_max = max(global_max, nums[i])
        note = ""
        if nums[i] < left_max:
            left_max = global_max
            pos = i + 1
            note = f"  -> {nums[i]} < left_max: mở rộng left tới i={i}, left_max={left_max}, pos={pos}"
        print(f"i={i}: x={nums[i]}, global_max={global_max}{note}")
    return pos

assert trace_one_pass([5, 0, 3, 8, 6]) == 3
print()
assert trace_one_pass([1, 1, 1, 0, 6, 12]) == 4

khởi tạo: left_max=5, global_max=5, pos=1
i=1: x=0, global_max=5  -> 0 < left_max: mở rộng left tới i=1, left_max=5, pos=2
i=2: x=3, global_max=5  -> 3 < left_max: mở rộng left tới i=2, left_max=5, pos=3
i=3: x=8, global_max=8
i=4: x=6, global_max=8

khởi tạo: left_max=1, global_max=1, pos=1
i=1: x=1, global_max=1
i=2: x=1, global_max=1
i=3: x=0, global_max=1  -> 0 < left_max: mở rộng left tới i=3, left_max=1, pos=4
i=4: x=6, global_max=6
i=5: x=12, global_max=12


### 8.2. Cách 2 (max tiền tố, min hậu tố) cho ví dụ 1

`pm = [5, 5, 5, 8, 8]`, `sm = [0, 0, 3, 6, 6]`. So sánh `pm[k - 1]` với `sm[k]`: `k = 1`: `5` vs `0` không; `k = 2`: `5` vs `3` không; `k = 3`: `5` vs `6` **có**.

In [13]:
nums = [5, 0, 3, 8, 6]
pm, sm = prefix_max(nums), suffix_min(nums)
print("pm =", pm, "| sm =", sm)
for k in range(1, len(nums)):
    print(f"k={k}: pm[{k - 1}]={pm[k - 1]} <= sm[{k}]={sm[k]} ? {pm[k - 1] <= sm[k]}")
assert partition_prefix_suffix(nums) == 3

pm = [5, 5, 5, 8, 8] | sm = [0, 0, 3, 6, 6]
k=1: pm[0]=5 <= sm[1]=0 ? False
k=2: pm[1]=5 <= sm[2]=3 ? False
k=3: pm[2]=5 <= sm[3]=6 ? True
k=4: pm[3]=8 <= sm[4]=6 ? False


### 8.3. Cách 4 (hiệu số đếm) cho ví dụ 1

`sorted = [0, 3, 5, 6, 8]`. `k = 1`: `nums[:1] = {5}` vs `{0}`: hiệu `{5: +1, 0: -1}`, 2 khóa khác 0. `k = 2`: thêm `0` và bớt `3`: `{5: +1, 0: 0, 3: -1}`, 2 khóa. `k = 3`: thêm `3` và bớt `5`: `{5: 0, 3: 0, 0: 0}`: **0 khóa** nên hợp lệ.

In [14]:
def trace_sorted_prefix(nums):
    s = sorted(nums)
    diff = defaultdict(int)
    print("sorted =", s)
    for k in range(1, len(nums)):
        diff[nums[k - 1]] += 1
        diff[s[k - 1]] -= 1
        nz = {v: c for v, c in diff.items() if c}
        print(f"k={k}: thêm {nums[k - 1]}, bớt {s[k - 1]}; hiệu khác 0: {nz}")
        if not nz:
            return k
    return -1

assert trace_sorted_prefix([5, 0, 3, 8, 6]) == 3

sorted = [0, 3, 5, 6, 8]
k=1: thêm 5, bớt 0; hiệu khác 0: {5: 1, 0: -1}
k=2: thêm 0, bớt 3; hiệu khác 0: {5: 1, 3: -1}
k=3: thêm 3, bớt 5; hiệu khác 0: {}


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (quét một lượt với hai biến): `O(n)` thời gian, `O(1)` bộ nhớ phụ, rất ngắn.

```python
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        left_max = global_max = nums[0]
        pos = 1
        for i in range(1, len(nums)):
            global_max = max(global_max, nums[i])
            if nums[i] < left_max:
                left_max = global_max
                pos = i + 1
        return pos
```

In [15]:
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        left_max = global_max = nums[0]
        pos = 1
        for i in range(1, len(nums)):
            global_max = max(global_max, nums[i])
            if nums[i] < left_max:
                left_max = global_max
                pos = i + 1
        return pos

sol = Solution()
for nums, expected in examples:
    got = sol.partitionDisjoint(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 0, 3, 8, 6] -> 3 OK
[1, 1, 1, 0, 6, 12] -> 4 OK


## 10. Vì sao đúng

**(a) Điều kiện điểm cắt.** `left = nums[:k]` hợp lệ khi và chỉ khi `max(nums[:k]) <= min(nums[k:])` (mọi `x` trái `<=` mọi `y` phải). Cách 2 kiểm tra đúng điều này bằng `pm[k - 1] <= sm[k]`, và trả `k` nhỏ nhất.

**(b) Cách 3 (quét một lượt).** Bất biến: sau khi xử lý `nums[0..i]`, `pos` là **điểm cắt hợp lệ nhỏ nhất** cho tiền tố `nums[0..i]` (xét tiền tố như mảng riêng, có thể `pos = i + 1` nghĩa là `right` rỗng tạm thời), `left_max = max(nums[0..pos-1])` và `global_max = max(nums[0..i])`.
   - Khởi tạo `i = 0`: `pos = 1`, `left_max = global_max = nums[0]`.
   - Bước `i`: nếu `nums[i] >= left_max` thì điểm cắt `pos` vẫn hợp lệ với `nums[i]` đặt ở `right` (`nums[i] >= left_max = max(left)`), và không có điểm cắt nhỏ hơn mới hợp lệ (cắt nhỏ hơn đã không hợp lệ ở tiền tố ngắn hơn, mà thêm phần tử vào `right` không làm một điểm cắt vốn không hợp lệ trở thành hợp lệ). Giữ nguyên.
   - Nếu `nums[i] < left_max`: `nums[i]` ở `right` sẽ phạm điều kiện với `left` hiện tại; mọi điểm cắt hợp lệ cho `nums[0..i]` phải có `left` chứa `nums[i]`, tức `left` dài tới ít nhất `i + 1`; điểm cắt nhỏ nhất như vậy là `i + 1` (cắt ở `i + 1` hợp lệ cho tiền tố vì `right` rỗng tạm thời), nên `pos = i + 1` và `left_max = max(nums[0..i]) = global_max`.
   Sau khi duyệt hết `nums`, `pos` là điểm cắt hợp lệ nhỏ nhất cho cả mảng; đề đảm bảo `pos <= n - 1` (có đáp án với `right` không rỗng): giả sử `pos = n` thì mọi điểm cắt thật có `left` dài `n` không hợp lệ (right rỗng); nhưng đề đảm bảo có điểm cắt hợp lệ `k <= n - 1`, mâu thuẫn với tính nhỏ nhất của `pos` vì mọi điểm cắt hợp lệ `k <= n - 1` cũng hợp lệ cho tiền tố, nên `pos <= k <= n - 1`.

**(c) Cách 4.** Nếu `max(nums[:k]) <= min(nums[k:])` thì `left` là `k` phần tử nhỏ nhất theo đa tập (không có phần tử bên phải nhỏ hơn bất kỳ phần tử bên trái), trùng đa tập `sorted(nums)[:k]`. Ngược lại nếu hai đa tập bằng nhau thì mọi phần tử `left` nằm trong `k` phần tử nhỏ nhất nên `<=` mọi phần tử ngoài chúng. Vậy hiệu đếm bằng `0` ⇔ điểm cắt hợp lệ.

**Kiểm chứng bằng máy.** Oracle **độc lập**: định nghĩa gốc "mọi cặp `(x, y)` với `x` trái và `y` phải thỏa `x <= y`" (Cách 1) trên mọi điểm cắt; với mảng lớn dùng Cách 2 (max tiền tố / min hậu tố) và Cách 4 (đa tập tiền tố đã sắp xếp), hai cách hoàn toàn khác Cách 3. Chỉ kiểm thử các đầu vào có đáp án (đề đảm bảo).

In [16]:
def has_answer(nums: List[int]) -> bool:
    n = len(nums)
    return any(all(x <= y for x in nums[:k] for y in nums[k:]) for k in range(1, n))

def gen_valid(rng, n, vmax):
    """Sinh mảng chắc chắn có đáp án: chọn điểm cắt k, phần trái <= M, phần phải >= M."""
    k = rng.randint(1, n - 1)
    M = rng.randint(0, vmax)
    left = [rng.randint(0, M) for _ in range(k)]
    right = [rng.randint(M, vmax) for _ in range(n - k)]
    return left + right

rng = random.Random(1915)
for _ in range(400):
    n = rng.randint(2, 9)
    nums = gen_valid(rng, n, rng.choice([3, 10, 100]))
    assert has_answer(nums)
    ref = partition_brute(nums)
    assert ref == partition_prefix_suffix(nums) == partition_one_pass(nums) == partition_sorted_prefix(nums), nums
print("định nghĩa gốc, max/min tiền hậu tố, một lượt và đa tập tiền tố khớp nhau trên 400 mảng có đáp án")

định nghĩa gốc, max/min tiền hậu tố, một lượt và đa tập tiền tố khớp nhau trên 400 mảng có đáp án


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng điểm cắt | `O(n^3)` (mọi cặp) hoặc `O(n^2)` | `O(1)` |
| 2. Max tiền tố, min hậu tố | `O(n)` | `O(n)` |
| 3. Quét một lượt | `O(n)` | `O(1)` |
| 4. Đa tập tiền tố đã sắp xếp | `O(n log n)` | `O(n)` |

Đo thử với `n = 10^5` (cỡ tối đa):

In [17]:
rng = random.Random(11)
n = 10 ** 5
nums = gen_valid(rng, n, 10 ** 6)
ref = partition_prefix_suffix(nums)
for name, fn in (("Cách 2 tiền tố/hậu tố", sol_prefix), ("Cách 3 một lượt", sol_one), ("Cách 4 đa tập", sol_sorted), ("nộp", Solution().partitionDisjoint)):
    t0 = time.perf_counter()
    got = fn(list(nums))
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:22} n={n}: {dt:8.1f} ms (đáp án {got})")
nums_small = gen_valid(rng, 400, 10 ** 6)
t0 = time.perf_counter()
partition_brute(nums_small)
print(f"{'Cách 1 (mọi cặp)':22} chỉ n=400: {1000 * (time.perf_counter() - t0):8.1f} ms")

Cách 2 tiền tố/hậu tố  n=100000:     18.1 ms (đáp án 59295)
Cách 3 một lượt        n=100000:      7.3 ms (đáp án 59295)


Cách 4 đa tập          n=100000:     50.3 ms (đáp án 59295)
nộp                    n=100000:      8.0 ms (đáp án 59295)
Cách 1 (mọi cặp)       chỉ n=400:      1.3 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle (định nghĩa gốc), test ngẫu nhiên có seed (đúng ràng buộc: luôn có đáp án) và tính chất. Mọi đầu vào được sinh hoặc lọc để **có đáp án** như đề đảm bảo.

In [18]:
def solutions_list():
    return {
        "prefix_suffix": partition_prefix_suffix,
        "one_pass": partition_one_pass,
        "sorted_prefix": partition_sorted_prefix,
        "sol_prefix": sol_prefix,
        "sol_one": sol_one,
        "sol_sorted": sol_sorted,
        "final": Solution().partitionDisjoint,
    }

def run_all(nums, expected=None, brute_limit=60):
    assert has_answer(nums) if len(nums) <= 60 else True, ("đầu vào không có đáp án", nums)
    ref = partition_brute(nums) if len(nums) <= brute_limit else partition_prefix_suffix(nums)
    for name, fn in solutions_list().items():
        assert fn(list(nums)) == ref, (name, nums)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", nums, expected, ref)
    return ref

# 12.1 ví dụ đề
for nums, expected in examples:
    assert run_all(nums, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1, 1], 1),                          # hai phần tử bằng nhau: left = [1]
    ([1, 2], 1),                          # tăng
    ([0, 0, 0], 1),                       # toàn bằng nhau: left nhỏ nhất là 1
    ([1, 2, 3, 4, 5], 1),                 # đã sắp xếp: left = [1]
    ([3, 1, 2, 4], 3),                    # [3,1,2] <= [4]
    ([2, 1, 3, 2, 5, 4, 6, 7], 2),        # [2,1] <= [3,2,5,4,6,7]
    ([4, 2, 5, 3, 6, 8, 7], 4),           # [4,2,5,3] <= [6,8,7]
    ([5, 0, 3, 8, 6], 3),                 # ví dụ 1
    ([1, 1, 1, 0, 6, 12], 4),             # ví dụ 2
    ([10 ** 6, 10 ** 6], 1),              # biên lớn, bằng nhau
    ([0, 10 ** 6], 1),                    # biên
    ([1, 0, 1], 2),                       # [1,0] <= [1]; dấu <= cho phép bằng nhau
    ([2, 2, 1, 2], 3),                    # [2,2,1] <= [2]
]
for nums, expected in edge:
    assert run_all(nums, expected) == expected, nums
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_strict(nums):                          # dùng < thay vì <= (không cho phép bằng nhau)
    n = len(nums)
    pm = [0] * n; pm[0] = nums[0]
    for i in range(1, n):
        pm[i] = max(pm[i - 1], nums[i])
    sm = [0] * n; sm[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        sm[i] = min(sm[i + 1], nums[i])
    for k in range(1, n):
        if pm[k - 1] < sm[k]:
            return k
    return -1

def wrong_no_update(nums):                       # mở rộng left nhưng quên cập nhật left_max = global_max
    left_max = global_max = nums[0]
    pos = 1
    for i in range(1, len(nums)):
        global_max = max(global_max, nums[i])
        if nums[i] < left_max:
            pos = i + 1                           # quên: left_max = global_max
    return pos

def wrong_sorted_max_only(nums):                 # so max(left) <= sorted[k] (bỏ sót phần tử nhỏ bên phải)
    s = sorted(nums)
    cur = nums[0]
    for k in range(1, len(nums)):
        cur = max(cur, nums[k - 1])
        if cur <= s[k]:
            return k
    return -1

def wrong_split_after_min(nums):                 # cắt ngay sau vị trí phần tử nhỏ nhất
    return nums.index(min(nums)) + 1

def wrong_first_not_greater(nums):               # cắt ở phần tử đầu tiên không lớn hơn phần tử đầu
    for i in range(1, len(nums)):
        if nums[i] >= nums[0]:
            return i
    return len(nums) - 1

def find_counter(wrong, tries=20000, seed=6):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 8)
        nums = gen_valid(rng, n, rng.choice([2, 5, 10]))
        if wrong(list(nums)) != partition_brute(nums):
            return nums
    return None

counters = {}
for name, fn in (("dùng < thay <=", wrong_strict), ("quên cập nhật left_max", wrong_no_update),
                 ("so max trái với sorted[k]", wrong_sorted_max_only), ("cắt sau phần tử nhỏ nhất", wrong_split_after_min),
                 ("cắt ở phần tử đầu tiên >= nums[0]", wrong_first_not_greater)):
    nums = find_counter(fn)
    assert nums is not None, name
    counters[name] = nums
    print(f"{name:34} nums={nums}: sai -> {fn(list(nums))}, đúng -> {partition_brute(nums)}")
    run_all(nums)
print("12.3 phản ví dụ: OK")

dùng < thay <=                     nums=[1, 1, 0, 0, 0, 1, 1, 6]: sai -> 7, đúng -> 5
quên cập nhật left_max             nums=[1, 3, 0, 1, 7]: sai -> 3, đúng -> 4
so max trái với sorted[k]          nums=[1, 1, 0, 0, 0, 1, 1, 6]: sai -> 3, đúng -> 5
cắt sau phần tử nhỏ nhất           nums=[1, 1, 0, 0, 0, 1, 1, 6]: sai -> 3, đúng -> 5
cắt ở phần tử đầu tiên >= nums[0]  nums=[1, 1, 0, 0, 0, 1, 1, 6]: sai -> 1, đúng -> 5
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ: mảng sinh có đáp án + mảng ngẫu nhiên lọc theo "có đáp án"; so mọi cách với định nghĩa gốc
rng = random.Random(4915)
count = 0
for _ in range(1500):
    n = rng.randint(2, 10)
    if rng.random() < 0.5:
        nums = gen_valid(rng, n, rng.choice([2, 5, 20, 10 ** 6]))
    else:
        nums = [rng.randint(0, rng.choice([2, 5, 20])) for _ in range(n)]
        if not has_answer(nums):
            continue
    run_all(nums)
    count += 1
print(f"12.4 ngẫu nhiên nhỏ ({count} ca có đáp án): OK")

# 12.5 trung bình: n tới 60 so với định nghĩa gốc; n tới 2000 so với Cách 2 và Cách 4
rng = random.Random(5915)
for _ in range(200):
    n = rng.randint(11, 60)
    run_all(gen_valid(rng, n, rng.choice([3, 50, 10 ** 6])))
for _ in range(60):
    n = rng.randint(61, 2000)
    run_all(gen_valid(rng, n, rng.choice([3, 50, 10 ** 6])))
print("12.5 ngẫu nhiên n tới 2000: OK")

# 12.6 lớn n = 10^5 (đúng ràng buộc) nhiều dạng
rng = random.Random(6915)
n = 10 ** 5
big = {
    "ngẫu nhiên có đáp án": gen_valid(rng, n, 10 ** 6),
    "đã sắp xếp": sorted(rng.randint(0, 10 ** 6) for _ in range(n)),
    "toàn bằng nhau": [5] * n,
    "tăng rồi một phần tử nhỏ ở cuối vùng trái": list(range(1000, 51000)) + [0] + list(range(51001, 100001)),
    "đáp án ở cuối": [10 ** 6] + [rng.randint(0, 10 ** 6 - 1) for _ in range(n - 2)] + [10 ** 6],
}
for name, nums in big.items():
    assert 2 <= len(nums) <= n and all(0 <= x <= 10 ** 6 for x in nums), name
    ref = partition_prefix_suffix(nums)
    for fname, fn in solutions_list().items():
        assert fn(list(nums)) == ref, (name, fname)
    print(f"  {name:44} đáp án {ref}")
print("12.6 n = 10^5: OK")

# 12.7 tính chất: đáp án hợp lệ (max(left) <= min(right)) và tối thiểu (mọi k nhỏ hơn không hợp lệ)
rng = random.Random(7915)
for _ in range(300):
    n = rng.randint(2, 40)
    nums = gen_valid(rng, n, rng.choice([3, 50, 1000]))
    k = Solution().partitionDisjoint(list(nums))
    assert 1 <= k <= n - 1 and max(nums[:k]) <= min(nums[k:])
    assert all(max(nums[:j]) > min(nums[j:]) for j in range(1, k))
print("12.7 tính chất (hợp lệ và tối thiểu): OK")

12.4 ngẫu nhiên nhỏ (1052 ca có đáp án): OK


12.5 ngẫu nhiên n tới 2000: OK


  ngẫu nhiên có đáp án                         đáp án 5614
  đã sắp xếp                                   đáp án 1
  toàn bằng nhau                               đáp án 1


  tăng rồi một phần tử nhỏ ở cuối vùng trái    đáp án 50001


  đáp án ở cuối                                đáp án 99999
12.6 n = 10^5: OK
12.7 tính chất (hợp lệ và tối thiểu): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` thay vì `<=`

Đề cho phép phần tử trái **bằng** phần tử phải (`<=`).

In [21]:
nums = counters["dùng < thay <="]
print("nums =", nums, "| sai ->", wrong_strict(list(nums)), "| đúng ->", Solution().partitionDisjoint(list(nums)))
assert wrong_strict(list(nums)) != Solution().partitionDisjoint(list(nums))
print("ví dụ gọn: [1, 1] ->", wrong_strict([1, 1]), "(sai) so với", Solution().partitionDisjoint([1, 1]))

nums = [1, 1, 0, 0, 0, 1, 1, 6] | sai -> 7 | đúng -> 5
ví dụ gọn: [1, 1] -> -1 (sai) so với 1


Vì sao sai: `max(left) == min(right)` vẫn hợp lệ (`[1] <= [1]`). **Đúng:** so sánh bằng `<=`.

### Lỗi 2 — Quên cập nhật `left_max` khi mở rộng `left`

Khi gặp phần tử nhỏ hơn `left_max`, `left` mở rộng tới hết vị trí hiện tại nên `left_max` phải trở thành **max toàn cục** đã gặp; quên điều này khiến các phần tử lớn đã bị đưa vào `left` không được tính.

In [22]:
nums = counters["quên cập nhật left_max"]
print("nums =", nums, "| sai ->", wrong_no_update(list(nums)), "| đúng ->", Solution().partitionDisjoint(list(nums)))
assert wrong_no_update(list(nums)) != Solution().partitionDisjoint(list(nums))

nums = [1, 3, 0, 1, 7] | sai -> 3 | đúng -> 4


Vì sao sai: sau khi `left` mở rộng, các phần tử lớn trong khoảng vừa nuốt vào `left` làm `max(left)` lớn hơn `left_max` cũ. **Đúng:** `left_max = global_max`.

### Lỗi 3 — Chỉ so `max(left)` với phần tử đã sắp xếp thứ `k`

Điều kiện `max(left) <= sorted[k]` **không đủ**: phần tử nhỏ bên phải vẫn có thể tồn tại nếu bên trái chứa nhiều phần tử bằng `sorted[k]`.

In [23]:
nums = counters["so max trái với sorted[k]"]
print("nums =", nums, "| sai ->", wrong_sorted_max_only(list(nums)), "| đúng ->", Solution().partitionDisjoint(list(nums)))
assert wrong_sorted_max_only(list(nums)) != Solution().partitionDisjoint(list(nums))

nums = [1, 1, 0, 0, 0, 1, 1, 6] | sai -> 3 | đúng -> 5


Vì sao sai: cần so `max(left)` với **mọi** phần tử bên phải (tức `min(right)`), không chỉ với phần tử thứ `k` của bản sắp xếp. **Đúng:** so đa tập tiền tố với tiền tố của bản sắp xếp (Cách 4), hoặc dùng `min(right)` (Cách 2).

### Lỗi 4 — Cắt dựa trên vị trí phần tử nhỏ nhất hoặc phần tử đầu

Những heuristic cục bộ không bảo đảm điều kiện toàn cục.

In [24]:
for name in ("cắt sau phần tử nhỏ nhất", "cắt ở phần tử đầu tiên >= nums[0]"):
    nums = counters[name]
    fn = wrong_split_after_min if "nhỏ nhất" in name else wrong_first_not_greater
    print(f"{name}: nums = {nums} | sai -> {fn(list(nums))} | đúng -> {Solution().partitionDisjoint(list(nums))}")
    assert fn(list(nums)) != Solution().partitionDisjoint(list(nums))

cắt sau phần tử nhỏ nhất: nums = [1, 1, 0, 0, 0, 1, 1, 6] | sai -> 3 | đúng -> 5
cắt ở phần tử đầu tiên >= nums[0]: nums = [1, 1, 0, 0, 0, 1, 1, 6] | sai -> 1 | đúng -> 5


Vì sao sai: điều kiện là về **mọi cặp** (`max(left) <= min(right)`), không phải về một phần tử đặc biệt. **Đúng:** max tiền tố so với min hậu tố.

### Lỗi 5 — Kiểm tra mọi cặp ở mỗi điểm cắt (`O(n^2)` hay `O(n^3)`)

Với `n = 10^5`, kiểm tra từng điểm cắt bằng quét là không khả thi (xem mục 5 và 11).

In [25]:
rng = random.Random(5)
for n in (200, 400, 800):
    nums = gen_valid(rng, n, 10 ** 6)
    t0 = time.perf_counter(); partition_brute(nums); t_brute = time.perf_counter() - t0
    t0 = time.perf_counter(); Solution().partitionDisjoint(list(nums)); t_fast = time.perf_counter() - t0
    print(f"n={n:4}: mọi cặp {1000 * t_brute:9.1f} ms | một lượt {1000 * t_fast:6.3f} ms")

n= 200: mọi cặp       0.3 ms | một lượt  0.016 ms
n= 400: mọi cặp       0.7 ms | một lượt  0.029 ms
n= 800: mọi cặp       3.4 ms | một lượt  0.054 ms


Vì sao chậm: mỗi điểm cắt tính lại `max`/`min` từ đầu. **Đúng:** tính trước max tiền tố và min hậu tố, hoặc quét một lượt.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia thành hai đoạn sao cho mọi phần tử trái `<=` mọi phần tử phải" ⇒ **`max(left) <= min(right)`** ⇒ **max tiền tố / min hậu tố** hoặc quét một lượt.
- **Quét một lượt:** giữ `left_max`, `global_max`, `pos`; khi thấy phần tử `< left_max` thì `pos = i + 1`, `left_max = global_max`.
- **Điểm cắt hợp lệ nhỏ nhất** ⇒ quét từ trái và dừng ở lần đầu thỏa.
- Cùng họ: Max Chunks To Make Sorted (769, 768), Best Time to Buy and Sell Stock (121), Product of Array Except Self (238), Trapping Rain Water (42).
- Test nên có: hai phần tử bằng nhau, mảng đã sắp xếp, toàn bằng nhau, phần tử nhỏ xuất hiện muộn kéo dài `left`, biên `10^6`.

## 15. Biến thể

1. **Đếm số điểm cắt hợp lệ** (bài tập 5): số `k` với `pm[k - 1] <= sm[k]`.
2. **Điểm cắt hợp lệ lớn nhất** (`left` dài nhất): quét từ phải sang trái hoặc dùng `pm`/`sm` và lấy `k` lớn nhất.
3. **Max Chunks To Make Sorted II (768):** số đoạn tối đa; dùng cùng `pm`/`sm`.
4. Bài liên quan: 768, 769.

Dưới đây là biến thể 2 (điểm cắt lớn nhất) kiểm bằng định nghĩa gốc.

In [26]:
def partition_largest_left(nums: List[int]) -> int:
    n = len(nums)
    pm = [0] * n
    pm[0] = nums[0]
    for i in range(1, n):
        pm[i] = max(pm[i - 1], nums[i])
    sm = [0] * n
    sm[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        sm[i] = min(sm[i + 1], nums[i])
    for k in range(n - 1, 0, -1):
        if pm[k - 1] <= sm[k]:
            return k
    return -1

def largest_left_brute(nums):
    return max(k for k in range(1, len(nums)) if all(x <= y for x in nums[:k] for y in nums[k:]))

rng = random.Random(15)
for _ in range(300):
    nums = gen_valid(rng, rng.randint(2, 9), rng.choice([3, 10, 100]))
    assert partition_largest_left(nums) == largest_left_brute(nums), nums
print("điểm cắt lớn nhất của ví dụ:", partition_largest_left([5, 0, 3, 8, 6]), partition_largest_left([1, 1, 1, 0, 6, 12]))

điểm cắt lớn nhất của ví dụ: 3 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng điểm cắt | Định nghĩa | `O(n^2)` - `O(n^3)` | `O(1)` | Oracle |
| 2. Max tiền tố, min hậu tố | `pm[k-1] <= sm[k]` | `O(n)` | `O(n)` | Dễ hiểu, mở rộng được |
| 3. Quét một lượt | Hai biến | `O(n)` | `O(1)` | **Cách nộp chính** |
| 4. Đa tập tiền tố đã sắp xếp | Hiệu số đếm | `O(n log n)` | `O(n)` | Kiểm chứng độc lập |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(prefix_max([5, 0, 3, 8, 6]))
print(suffix_min([5, 0, 3, 8, 6]))
print(valid_split([5, 0, 3, 8, 6], 3), valid_split([5, 0, 3, 8, 6], 4))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [4, 2, 5, 3, 6, 8, 7]`: ghi độ dài `left` nhỏ nhất và `left`, `right` tương ứng.

In [28]:
# Bài 2: điền kết quả tính tay
hand_k = None
hand_left = None
hand_right = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách max tiền tố và min hậu tố.

In [29]:
def exercise_3_prefix_suffix(nums: List[int]) -> int:
    n = len(nums)
    pm = [0] * n
    pm[0] = nums[0]
    # TODO 1: điền pm[i] = max(pm[i - 1], nums[i])
    sm = [0] * n
    sm[-1] = nums[-1]
    # TODO 2: điền sm[i] = min(sm[i + 1], nums[i]) (duyệt lùi)
    # TODO 3: tìm k nhỏ nhất thỏa pm[k - 1] <= sm[k]
    return -1


# for nums, expected in examples:
#     assert exercise_3_prefix_suffix(nums) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_one_pass(nums)` bằng quét một lượt với hai biến.

In [30]:
def exercise_4_one_pass(nums: List[int]) -> int:
    # TODO
    pass


# for nums, expected in examples:
#     assert exercise_4_one_pass(nums) == expected

### Bài 5 — Mini challenge: đếm số điểm cắt hợp lệ (khó)

Viết `exercise_5_count(nums)` trả về **số điểm cắt `k`** (`1 <= k <= n - 1`) sao cho `max(nums[:k]) <= min(nums[k:])`. Không nhất thiết có đáp án: trả `0` nếu không có.

```python
exercise_5_count([5, 0, 3, 8, 6])     -> 1      # chỉ k = 3
exercise_5_count([1, 2, 3, 4, 5])     -> 4      # mọi k = 1..4
exercise_5_count([1, 1, 1])           -> 2      # k = 1, 2 (bằng nhau vẫn hợp lệ)
exercise_5_count([2, 1])              -> 0
```

In [31]:
def exercise_5_count(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_count([5, 0, 3, 8, 6]) == 1
# assert exercise_5_count([1, 2, 3, 4, 5]) == 4
# assert exercise_5_count([1, 1, 1]) == 2
# assert exercise_5_count([2, 1]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: max tiền tố tăng dần, min hậu tố tăng dần (từ trái sang phải).
- Mức 2: max tiền tố của `[5,0,3,8,6]`: `5,5,5,8,8`; min hậu tố: `0,0,3,6,6`.
- Mức 3: `[5, 5, 5, 8, 8]`, `[0, 0, 3, 6, 6]`, `True False`.

**Bài 2**
- Mức 1: thử `k = 1, 2, 3, 4`.
- Mức 2: `k = 4`: `left = [4,2,5,3]`, `max = 5`; `right = [6,8,7]`, `min = 6`.
- Mức 3: `k = 4`, `left = [4,2,5,3]`, `right = [6,8,7]`.

**Bài 3**
- Mức 1: `for i in range(1, n): pm[i] = max(pm[i - 1], nums[i])`.
- Mức 2: `for i in range(n - 2, -1, -1): sm[i] = min(sm[i + 1], nums[i])`.
- Mức 3: `for k in range(1, n): if pm[k - 1] <= sm[k]: return k`.

**Bài 4**
- Mức 1: khởi tạo `left_max = global_max = nums[0]`, `pos = 1`.
- Mức 2: cập nhật `global_max`; nếu `nums[i] < left_max`.
- Mức 3: `left_max = global_max`; `pos = i + 1`.

**Bài 5**
- Mức 1: dùng `pm` và `sm`.
- Mức 2: đếm `k` với `pm[k - 1] <= sm[k]`.
- Mức 3: `sum(1 for k in range(1, n) if pm[k - 1] <= sm[k])`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(prefix_max([5, 0, 3, 8, 6]))
print(suffix_min([5, 0, 3, 8, 6]))
print(valid_split([5, 0, 3, 8, 6], 3), valid_split([5, 0, 3, 8, 6], 4))

assert prefix_max([5, 0, 3, 8, 6]) == [5, 5, 5, 8, 8]
assert suffix_min([5, 0, 3, 8, 6]) == [0, 0, 3, 6, 6]
assert (valid_split([5, 0, 3, 8, 6], 3), valid_split([5, 0, 3, 8, 6], 4)) == (True, False)

[5, 5, 5, 8, 8]
[0, 0, 3, 6, 6]
True False


### Đáp án Bài 2

`k = 4`: `left = [4, 2, 5, 3]` (max `5`), `right = [6, 8, 7]` (min `6`).

In [33]:
nums = [4, 2, 5, 3, 6, 8, 7]
hand_k_solution = 4
hand_left_solution, hand_right_solution = [4, 2, 5, 3], [6, 8, 7]
assert Solution().partitionDisjoint(list(nums)) == hand_k_solution == partition_brute(nums)
assert nums[:hand_k_solution] == hand_left_solution and nums[hand_k_solution:] == hand_right_solution
assert max(hand_left_solution) <= min(hand_right_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_prefix_suffix_solution(nums: List[int]) -> int:
    n = len(nums)
    pm = [0] * n
    pm[0] = nums[0]
    for i in range(1, n):
        pm[i] = max(pm[i - 1], nums[i])
    sm = [0] * n
    sm[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        sm[i] = min(sm[i + 1], nums[i])
    for k in range(1, n):
        if pm[k - 1] <= sm[k]:
            return k
    return -1

for nums, expected in examples:
    assert exercise_3_prefix_suffix_solution(nums) == expected
rng = random.Random(3)
for _ in range(300):
    nums = gen_valid(rng, rng.randint(2, 10), rng.choice([3, 20, 1000]))
    assert exercise_3_prefix_suffix_solution(nums) == partition_brute(nums)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_one_pass_solution(nums: List[int]) -> int:
    left_max = global_max = nums[0]
    pos = 1
    for i in range(1, len(nums)):
        global_max = max(global_max, nums[i])
        if nums[i] < left_max:
            left_max = global_max
            pos = i + 1
    return pos

for nums, expected in examples:
    assert exercise_4_one_pass_solution(nums) == expected
rng = random.Random(4)
for _ in range(300):
    nums = gen_valid(rng, rng.randint(2, 10), rng.choice([3, 20, 1000]))
    assert exercise_4_one_pass_solution(nums) == partition_brute(nums)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_count_solution(nums: List[int]) -> int:
    n = len(nums)
    pm = [0] * n
    pm[0] = nums[0]
    for i in range(1, n):
        pm[i] = max(pm[i - 1], nums[i])
    sm = [0] * n
    sm[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        sm[i] = min(sm[i + 1], nums[i])
    return sum(1 for k in range(1, n) if pm[k - 1] <= sm[k])

def count_brute(nums):
    return sum(1 for k in range(1, len(nums)) if all(x <= y for x in nums[:k] for y in nums[k:]))

assert exercise_5_count_solution([5, 0, 3, 8, 6]) == 1
assert exercise_5_count_solution([1, 2, 3, 4, 5]) == 4
assert exercise_5_count_solution([1, 1, 1]) == 2
assert exercise_5_count_solution([2, 1]) == 0
rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(0, 5) for _ in range(rng.randint(2, 9))]
    assert exercise_5_count_solution(nums) == count_brute(nums), nums
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (quét một lượt với hai biến)

```python
class Solution:
    def partitionDisjoint(self, nums: List[int]) -> int:
        left_max = global_max = nums[0]       # max của left hiện tại / max toàn cục đã duyệt
        pos = 1                                # độ dài left hiện tại
        for i in range(1, len(nums)):
            global_max = max(global_max, nums[i])
            if nums[i] < left_max:             # phần tử nhỏ hơn max(left) phải thuộc left
                left_max = global_max          # left mở rộng tới hết i: max(left) = max toàn cục
                pos = i + 1
        return pos
```

Công thức nhớ nhanh: điểm cắt `k` hợp lệ ⇔ `max(nums[:k]) <= min(nums[k:])`; quét một lượt: `nums[i] < left_max` ⇒ `pos = i + 1`, `left_max = global_max`.

### Checklist trước khi submit

- [ ] So sánh bằng `<=` (không `<`)?
- [ ] Khi mở rộng `left`, cập nhật `left_max = global_max` (không giữ `left_max` cũ)?
- [ ] Trả về **độ dài** `left` (`pos`), không phải chỉ số?
- [ ] Không dùng `O(n^2)` với `n = 10^5`?
- [ ] Đã thử: `[1,1]`, mảng đã sắp xếp, toàn bằng nhau, phần tử nhỏ xuất hiện muộn, biên `10^6`?
- [ ] Tên method đúng `partitionDisjoint(self, nums)` và trả `int`?